# NASA C-MAPSS FD002 — 고급 버전: 운영조건(Operating Condition) 기반 정규화 + 강한 시퀀스 모델

이 노트북은 FD002에서 점수 향상에 자주 도움이 되는 **운영조건 기반 정규화**를 포함합니다.

핵심 아이디어:
- FD002는 여러 operating conditions(조건 조합)에서 측정됩니다.
- 센서 분포가 조건에 따라 달라서 **전역(StandardScaler 1개)** 로만 정규화하면 모델이 헷갈릴 수 있습니다.
- 그래서 `op1~op3`(운영 세팅) 기반으로 **KMeans 클러스터링**을 하고,
  각 클러스터별로 **센서/조건을 따로 스케일링**합니다.

또한 모델은 **CNN + BiLSTM + Attention**을 사용합니다.
- (추가) 약간의 regularization + early stopping + lr schedule 포함

---

## 파일 구조
```
data/
  train_FD002.txt
  test_FD002.txt
  RUL_FD002.txt
```

In [ ]:
# ====== 1) 라이브러리 ======
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, mean_absolute_error

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

In [ ]:
# ====== 2) 데이터 로드 ======
DATA_DIR = "data"   # <-- 필요시 변경

TRAIN_PATH = os.path.join(DATA_DIR, "train_FD002.txt")
TEST_PATH  = os.path.join(DATA_DIR, "test_FD002.txt")
RUL_PATH   = os.path.join(DATA_DIR, "RUL_FD002.txt")

assert os.path.exists(TRAIN_PATH), f"파일 없음: {TRAIN_PATH}"
assert os.path.exists(TEST_PATH),  f"파일 없음: {TEST_PATH}"
assert os.path.exists(RUL_PATH),   f"파일 없음: {RUL_PATH}"

train_raw = pd.read_csv(TRAIN_PATH, delim_whitespace=True, header=None)
test_raw  = pd.read_csv(TEST_PATH,  delim_whitespace=True, header=None)
rul_true  = pd.read_csv(RUL_PATH,   delim_whitespace=True, header=None, names=["RUL_true"])

print("raw shapes:", train_raw.shape, test_raw.shape, rul_true.shape)
train_raw.head()

In [ ]:
# ====== 3) 컬럼 정리 ======
def fix_columns(df):
    df = df.copy()
    df = df.dropna(axis=1, how="all")
    df = df.iloc[:, :26]  # 안전하게 앞 26개 사용
    cols = ["unit", "cycle", "op1", "op2", "op3"] + [f"s{i}" for i in range(1, 22)]
    df.columns = cols
    return df

train = fix_columns(train_raw)
test  = fix_columns(test_raw)

train.head()

## 4) RUL 라벨 만들기 + Capping

- `RUL = max_cycle(unit) - cycle`
- 학습 안정화를 위해 RUL 상한 적용 (보통 100~130 사이 튜닝)

In [ ]:
RUL_CAP = 125  # 튜닝 포인트: 100, 125, 130 등 시도

max_cycle = train.groupby("unit")["cycle"].max().rename("max_cycle")
train = train.merge(max_cycle, on="unit", how="left")
train["RUL"] = train["max_cycle"] - train["cycle"]
train.drop(columns=["max_cycle"], inplace=True)
train["RUL_capped"] = train["RUL"].clip(upper=RUL_CAP)

train[["unit","cycle","RUL","RUL_capped"]].head()

## 5) 센서 저분산 제거

거의 변하지 않는 센서는 제거(모델 방해 가능).

In [ ]:
sensor_cols = [c for c in train.columns if c.startswith("s")]
sensor_var = train[sensor_cols].var().sort_values()

VAR_TH = 1e-6
keep_sensors = sensor_var[sensor_var > VAR_TH].index.tolist()
drop_sensors = [c for c in sensor_cols if c not in keep_sensors]

print("drop sensors:", drop_sensors)
print("keep sensors:", len(keep_sensors))

feature_cols = ["op1","op2","op3"] + keep_sensors
target_col = "RUL_capped"

## 6) 엔진 단위 Train/Val split (누수 방지)

In [ ]:
units = train["unit"].unique()
train_units, val_units = train_test_split(units, test_size=0.2, random_state=SEED)

train_df = train[train["unit"].isin(train_units)].copy()
val_df   = train[train["unit"].isin(val_units)].copy()

print("train units:", len(train_units), "val units:", len(val_units))
print("rows:", train_df.shape, val_df.shape)

## 7) (핵심) 운영조건 기반 클러스터링 + 클러스터별 스케일링

### 왜 KMeans?
FD002는 조건이 이산적으로 섞여 있어 보통 **조건별로 센서 평균/분산이 달라집니다.**
`op1~op3`를 입력으로 KMeans를 돌려 조건 군집을 만든 뒤,
각 군집마다 `StandardScaler`를 따로 학습합니다.

> 실전 팁  
> - 클러스터 수 `K`는 4~8 사이에서 튜닝해보면 좋습니다.  
> - FD002에서 흔히 `K=6`이 무난하게 잘 맞는 경우가 많습니다.

In [ ]:
# ====== 7) KMeans on op settings ======
K = 6  # 튜닝 포인트: 4,5,6,7,8...

op_cols = ["op1","op2","op3"]

kmeans = KMeans(n_clusters=K, random_state=SEED, n_init="auto")
kmeans.fit(train_df[op_cols])

train_df["op_cluster"] = kmeans.predict(train_df[op_cols])
val_df["op_cluster"]   = kmeans.predict(val_df[op_cols])
test["op_cluster"]     = kmeans.predict(test[op_cols])

train_df["op_cluster"].value_counts().sort_index()

In [ ]:
# ====== 8) cluster-wise scaler ======
# (주의) scaler는 train_df 기준으로만 fit. val/test에는 transform만.
scalers = {}

# 각 클러스터별로 스케일러 학습
for c in range(K):
    df_c = train_df[train_df["op_cluster"]==c]
    sc = StandardScaler()
    sc.fit(df_c[feature_cols])
    scalers[c] = sc

def transform_clusterwise(df):
    df = df.copy()
    for c in range(K):
        idx = (df["op_cluster"]==c)
        if idx.any():
            df.loc[idx, feature_cols] = scalers[c].transform(df.loc[idx, feature_cols])
    return df

train_df_s = transform_clusterwise(train_df)
val_df_s   = transform_clusterwise(val_df)
test_s     = transform_clusterwise(test)

train_df_s.head()

## 9) 시퀀스(윈도우) 만들기

- window length: 30~50 추천
- label: 윈도우 마지막 시점 RUL
- 학습 데이터 수가 너무 많으면 step=2로 줄여도 성능이 유지되거나 좋아지기도 합니다.

In [ ]:
WINDOW = 40   # 튜닝 포인트: 30, 40, 50
STEP   = 1    # 튜닝 포인트: 1, 2

def make_windows(df, window=40, step=1, is_train=True):
    X_list, y_list, uid_list = [], [], []
    for uid, g in df.groupby("unit"):
        g = g.sort_values("cycle")
        data = g[feature_cols].values.astype(np.float32)
        if is_train:
            y = g[target_col].values.astype(np.float32)
            for start in range(0, len(g)-window+1, step):
                end = start + window
                X_list.append(data[start:end])
                y_list.append(y[end-1])
                uid_list.append(uid)
        else:
            if len(g) >= window:
                X_list.append(data[-window:])
            else:
                pad = np.repeat(data[:1], window-len(g), axis=0)
                X_list.append(np.vstack([pad, data]).astype(np.float32))
            uid_list.append(uid)
    X = np.stack(X_list)
    if is_train:
        return X, np.array(y_list, dtype=np.float32), np.array(uid_list)
    return X, np.array(uid_list)

X_train, y_train, _ = make_windows(train_df_s, window=WINDOW, step=STEP, is_train=True)
X_val,   y_val,   _ = make_windows(val_df_s,   window=WINDOW, step=STEP, is_train=True)
X_test,  uid_test   = make_windows(test_s,     window=WINDOW, step=1,    is_train=False)

X_train.shape, X_val.shape, X_test.shape

## 10) 모델: CNN + BiLSTM + Attention (강한 베이스라인)

- Conv1D 블록 2개(패턴 추출)
- BiLSTM (시간 의존)
- 간단 Attention(가중합)
- Dense 회귀

학습 안정화:
- BatchNorm, Dropout
- EarlyStopping + ReduceLROnPlateau

In [ ]:
def build_model(window, n_feat):
    inp = keras.Input(shape=(window, n_feat))

    x = layers.Conv1D(96, 3, padding="same", activation="relu")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Conv1D(96, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Bidirectional(layers.LSTM(96, return_sequences=True))(x)
    x = layers.Dropout(0.25)(x)

    # attention
    attn = layers.Dense(1, activation="tanh")(x)     # (B,T,1)
    attn = layers.Flatten()(attn)                   # (B,T)
    attn = layers.Activation("softmax")(attn)       # (B,T)
    attn = layers.RepeatVector(192)(attn)           # (B,192,T)
    attn = layers.Permute([2,1])(attn)              # (B,T,192)
    x = layers.Multiply()([x, attn])                # (B,T,192)
    x = layers.Lambda(lambda t: tf.reduce_sum(t, axis=1))(x)  # (B,192)

    x = layers.Dense(192, activation="relu")(x)
    x = layers.Dropout(0.25)(x)
    out = layers.Dense(1)(x)

    model = keras.Model(inp, out)
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="mse",
        metrics=[keras.metrics.RootMeanSquaredError(name="rmse"),
                 keras.metrics.MeanAbsoluteError(name="mae")]
    )
    return model

model = build_model(WINDOW, X_train.shape[-1])
model.summary()

In [ ]:
BATCH = 256
EPOCHS = 100

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_rmse", patience=12, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_rmse", factor=0.5, patience=6, min_lr=1e-5),
]

hist = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
plt.figure()
plt.plot(hist.history["rmse"], label="train_rmse")
plt.plot(hist.history["val_rmse"], label="val_rmse")
plt.title("RMSE over epochs")
plt.xlabel("epoch"); plt.ylabel("rmse"); plt.legend()
plt.show()

## 11) 평가: RMSE / MAE / NASA Score

NASA score는 과소추정(늦게 고장 예측)에 더 큰 패널티를 주는 비대칭 점수로 널리 사용됩니다.

In [ ]:
def nasa_score(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    d = y_pred - y_true
    s = np.where(d < 0, np.exp(-d / 13.0) - 1.0, np.exp(d / 10.0) - 1.0)
    return float(np.sum(s))

val_pred = model.predict(X_val, batch_size=1024).reshape(-1)

val_rmse = math.sqrt(mean_squared_error(y_val, val_pred))
val_mae  = mean_absolute_error(y_val, val_pred)
val_sc   = nasa_score(y_val, val_pred)

print("VAL RMSE:", val_rmse)
print("VAL MAE :", val_mae)
print("VAL NASA score:", val_sc)

In [ ]:
plt.figure()
plt.scatter(y_val, val_pred, s=8)
plt.xlabel("True RUL"); plt.ylabel("Pred RUL")
plt.title("Validation: True vs Pred")
plt.show()

plt.figure()
err = val_pred - y_val
plt.hist(err, bins=60)
plt.title("Validation error (pred-true)")
plt.xlabel("error"); plt.ylabel("count")
plt.show()

## 12) Test 평가 (test_FD002 + RUL_FD002)

엔진별 마지막 window 1개로 예측합니다.

In [ ]:
test_pred = model.predict(X_test, batch_size=1024).reshape(-1)

pred_df = pd.DataFrame({"unit": uid_test, "RUL_pred": test_pred}).sort_values("unit").reset_index(drop=True)
true_df = pd.DataFrame({"unit": np.sort(test["unit"].unique()), "RUL_true": rul_true["RUL_true"].values})
test_eval = true_df.merge(pred_df, on="unit", how="left")

test_rmse = math.sqrt(mean_squared_error(test_eval["RUL_true"], test_eval["RUL_pred"]))
test_mae  = mean_absolute_error(test_eval["RUL_true"], test_eval["RUL_pred"])
test_sc   = nasa_score(test_eval["RUL_true"], test_eval["RUL_pred"])

print("TEST RMSE:", test_rmse)
print("TEST MAE :", test_mae)
print("TEST NASA score:", test_sc)

test_eval.head()

In [ ]:
plt.figure()
plt.scatter(test_eval["RUL_true"], test_eval["RUL_pred"], s=10)
plt.xlabel("True RUL"); plt.ylabel("Pred RUL")
plt.title("Test: True vs Pred (FD002)")
plt.show()

## 13) 더 점수 올리는 “실전 튜닝” 체크리스트

아래 4개만 조합해도 점수가 꽤 변합니다.

1) **K (조건 클러스터 수)**: `4~8`  
2) **WINDOW**: `30/40/50`  
3) **RUL_CAP**: `100/125/130`  
4) **STEP**: `1/2` (데이터 수/과적합 조절)

원하면 내가 위 파라미터들을 자동으로 여러 조합 돌려서  
`(val RMSE, val NASA score)` 기준으로 **최적 조합 찾는 튜닝 코드(간단 Grid Search)** 도 같이 붙여줄게.